In [ ]:

from dataclasses import dataclass, field

class Term:
    def __add__(self, other):
        return App("+", (self, other))

@dataclass(frozen=True)
class App(Term):
    f: str
    args : tuple[Term,...]
@dataclass(frozen=True)
class Var(Term):
    name: str

def fv(term : Term) -> set[Var]:
    match term:
        case Var(_):
            return {term}
        case App(f, args):
            return set.union(*(fv(arg) for arg in args))

def rename(term : Term, subst : dict[Var,Var]):
    match term:
        case Var(_):
            return subst.get(term, term)
        case App(f, args):
            return App(f, tuple(rename(arg, subst) for arg in args))

@dataclass
class TermSet:
    #public_slots : set[str] = field(default_factory=set)
    # covariant_vats
    noninvariant_vars : set[Var]
    terms : set[Term]

    def __post_init__(self):
        # every fixed var should be in a term, or something has gone wrong
        assert all(any(v in fv(t) for t in self.terms) for v in self.noninvariant_vars)



    def rename(self, subst : dict[Var,Var]):
        return TermSet({subst.get(v,v) for v in self.noninvariant_vars}, {rename(term, subst) for term in self.terms})
    
    @staticmethod
    def sing(term : Term):
        return TermSet(fv(term), {term})

    def __or__(self, other):
        assert self.noninvariant_vars == other.noninvariant_vars
        return TermSet(self.noninvariant_vars, self.terms | other.terms)
    
    def __and__(self, other):
        # unify?
        ...

    def fv(self):
        # fv is not breaking a boundary. Variables 
        return set().union(*(fv(term) for term in self.terms))
    def __eq__(self, other):
        ...
        assert self.noninvariant_vars == other.noninvariant_vars
        # TODO: discover permutation of 

x,y,z = Var("x"), Var("y"), Var("z")

TermSet({x,y}, {x+y}).rename({x:z})

TermSet.sing(x+y)




TermSet(noninvariant_vars={Var(name='y'), Var(name='x')}, terms={App(f='+', args=(Var(name='x'), Var(name='y')))})